# 02_image_embeddings
This notebook creates visual feature embeddings from product images.


In [1]:
import sys

print(sys.executable)

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Scripts\python.exe


In [2]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Torch: 2.14.0+cpu
Torchvision: 0.29.0+cpu


In [3]:
import torch
import torch.nn as nn

from torchvision.models import resnet50, ResNet50_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)

# Remove final ImageNet classification layer
model.fc = nn.Identity()

model = model.to(device)
model.eval()

print("Device:", device)
print("ResNet50 loaded successfully")

Device: cpu
ResNet50 loaded successfully


In [4]:
import pandas as pd
import ast
import requests
import io

from PIL import Image

products = pd.read_csv(
    "../data/processed/products.csv"
)

print("Number of products:", len(products))
print("Columns:")
print(products.columns.tolist())

Number of products: 5000
Columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'product_id', 'bought_together', 'subtitle', 'author']


In [5]:
print(products.columns.tolist())

['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'product_id', 'bought_together', 'subtitle', 'author']


In [6]:
print(products["images"].iloc[0])

{'hi_res': array(['https://m.media-amazon.com/images/I/61KIZjb54AL._AC_UL1500_.jpg',
       'https://m.media-amazon.com/images/I/61OkTrsW88L._AC_UL1500_.jpg',
       'https://m.media-amazon.com/images/I/811lbVuVIAL._AC_UL1500_.jpg',
       'https://m.media-amazon.com/images/I/71wa-UQGlYL._AC_UL1500_.jpg',
       'https://m.media-amazon.com/images/I/61tspZIpHqL._AC_UL1500_.jpg',
       'https://m.media-amazon.com/images/I/71oVECB3VYL._AC_UL1500_.jpg'],
      dtype=object), 'large': array(['https://m.media-amazon.com/images/I/31zPYU+UkVL._AC_.jpg',
       'https://m.media-amazon.com/images/I/31LYNX5KQ6L._AC_.jpg',
       'https://m.media-amazon.com/images/I/51dD8Ga+wQL._AC_.jpg',
       'https://m.media-amazon.com/images/I/41kM3Cn3ERL._AC_.jpg',
       'https://m.media-amazon.com/images/I/41ZzITDO+OL._AC_.jpg',
       'https://m.media-amazon.com/images/I/51jTzfIP3dL._AC_.jpg'],
      dtype=object), 'thumb': array(['https://m.media-amazon.com/images/I/31zPYU+UkVL._AC_SR38,50_.jpg',
      

In [7]:
import re

def extract_image_url(value):
    if pd.isna(value):
        return None

    value = str(value)

    urls = re.findall(
        r'https?://[^\s\'"\],]+',
        value
    )

    if len(urls) > 0:
        return urls[0]

    return None


products["image_url"] = products["images"].apply(
    extract_image_url
)

print(
    products[
        ["product_id", "title", "image_url"]
    ].head(10).to_string()
)

print(
    "Products with image URLs:",
    products["image_url"].notna().sum()
)

   product_id                                                                                                             title                                                        image_url
0  B09X1MRDN6      BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover Up Shirt Dress with Pockets  https://m.media-amazon.com/images/I/61KIZjb54AL._AC_UL1500_.jpg
1  B073C4Q7W8                                                                    Merrell Work Moab 2 Vent Waterproof SR Boulder  https://m.media-amazon.com/images/I/81oltcdwYSL._AC_UL1500_.jpg
2  B0944VG4Y4                                                                                       SAS Women's, Relaxed Sandal  https://m.media-amazon.com/images/I/61Q8PVmaltL._AC_UL1500_.jpg
3  B08JGGF5TJ                                                SheIn Women's Basic Stretch Plaid Mini Bodycon A-Line Pencil Skirt  https://m.media-amazon.com/images/I/81PEsSG-hPL._AC_UL1500_.jpg
4  B00ZQMM6BI                      

In [8]:
valid_products = products[
    products["image_url"].notna()
].copy()

print("Valid image URLs:", len(valid_products))

Valid image URLs: 4998


In [9]:
url = valid_products["image_url"].iloc[0]

print(url)


https://m.media-amazon.com/images/I/61KIZjb54AL._AC_UL1500_.jpg


In [10]:
response = requests.get(
    url,
    timeout=15
)

print("Status code:", response.status_code)
print("Downloaded bytes:", len(response.content))

Status code: 200
Downloaded bytes: 68907


In [11]:
image = Image.open(
    io.BytesIO(response.content)
).convert("RGB")

print("Image size:", image.size)

Image size: (683, 1500)


In [12]:
preprocess = weights.transforms()

input_tensor = preprocess(
    image
).unsqueeze(0).to(device)

with torch.no_grad():
    embedding = model(input_tensor)

print("Embedding shape:", embedding.shape)

Embedding shape: torch.Size([1, 2048])


In [13]:
from tqdm.auto import tqdm

image_embeddings = []
image_product_ids = []

for _, row in tqdm(
    products.iterrows(),
    total=len(products)
):

    product_id = row["product_id"]
    url = row["image_url"]

    if not isinstance(url, str):
        continue

    try:

        response = requests.get(
            url,
            timeout=10
        )

        if response.status_code != 200:
            continue

        image = Image.open(
            io.BytesIO(response.content)
        ).convert("RGB")

        tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = model(tensor)

        embedding = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )

        image_embeddings.append(embedding)
        image_product_ids.append(product_id)

    except Exception:
        continue

print(
    "Images processed:",
    len(image_embeddings)
)

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 5000/5000 [1:48:49<00:00,  1.31s/it]  

Images processed: 4996


In [15]:
import numpy as np
import pandas as pd
import os

In [16]:
image_embeddings = np.asarray(
    image_embeddings,
    dtype=np.float32
)

print(
    "Image embedding shape:",
    image_embeddings.shape
)

Image embedding shape: (4996, 2048)


In [17]:
os.makedirs(
    "../data/embeddings",
    exist_ok=True
)

np.save(
    "../data/embeddings/image_embeddings.npy",
    image_embeddings
)

pd.DataFrame({
    "product_id": image_product_ids
}).to_csv(
    "../data/embeddings/image_product_ids.csv",
    index=False
)

print("Image embeddings saved successfully.")

Image embeddings saved successfully.


In [18]:
saved = np.load(
    "../data/embeddings/image_embeddings.npy"
)

print("Saved shape:", saved.shape)
print("Data type:", saved.dtype)

Saved shape: (4996, 2048)
Data type: float32
